# Feature Exploration Update — Sample Frames, Feature Viz, and New Directions

This notebook extends `feature_extraction.ipynb` for the project update:

1. A per-class sample image + feature visualization panel (raw frame, HOG, YOLO detections, Hough/vanishing-point, road segmentation) for all 5 maneuver classes.
2. t-SNE and UMAP projections alongside the existing PCA, to get a second (nonlinear) read on how separable each feature set is by maneuver class.
3. A new complex feature: a dense CNN embedding pulled from the YOLOv8s backbone (not just detection counts), using `ultralytics`' built-in `embed` support.
4. A written summary of what we're seeing and where to focus tinkering effort next.

Run this with the `281-s2-group2` kernel — it needs `standard_e2e`, `ultralytics`, and the feature arrays already produced by `feature_extraction.ipynb`.

> `pip install umap-learn` if you haven't already (only new dependency vs. the existing environment.yml).

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.makedirs('outputs', exist_ok=True)
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter
from standard_e2e import Modality
import time

TRAIN_DIR     = '../data/processed/waymo_e2e/training/'
MANIFEST_PATH = '../data/train_manifest.json'
FEATURES_DIR  = '../data/processed/waymo_e2e/features/'

CLASSES = ['straight', 'left-turn', 'right-turn', 'lane-change-left', 'lane-change-right']
COLORS = {
    'straight': 'steelblue', 'left-turn': 'tomato', 'right-turn': 'green',
    'lane-change-left': 'purple', 'lane-change-right': 'orange',
}

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

examples = {}
for seq_id, entry in manifest.items():
    label = entry['label']
    if label in CLASSES and label not in examples:
        data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
        examples[label] = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    if len(examples) == len(CLASSES):
        break

print(Counter(v['label'] for v in manifest.values()))

## 1. Per-class sample image + feature visualization panel

Five rows per class: raw frame, HOG, YOLO detections, Hough lines + vanishing point, road segmentation.
This is the literal deliverable for the project update — reuses the exact feature functions from
`feature_extraction.ipynb` so the visuals stay consistent with what's already been reviewed.

In [ ]:
from skimage.feature import hog
from ultralytics import YOLO

HOG_PARAMS = {'orientations': 9, 'pixels_per_cell': (16, 16), 'cells_per_block': (2, 2), 'channel_axis': -1}

def extract_hog(img):
    return hog(img, visualize=True, **HOG_PARAMS)

def detect_edges_and_lines(img, canny_low=30, canny_high=100, hough_threshold=15,
                            min_line_length=20, max_line_gap=15, road_region_frac=0.55):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape
    road_mask = np.zeros_like(gray); road_mask[int(H * road_region_frac):, :] = 1
    edges = cv2.Canny(gray, canny_low, canny_high)
    edges_masked = edges * road_mask
    lines_raw = cv2.HoughLinesP(edges_masked, rho=1, theta=np.pi / 180, threshold=hough_threshold,
                                 minLineLength=min_line_length, maxLineGap=max_line_gap)
    lines = []
    if lines_raw is not None:
        for line in lines_raw:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if 20 < angle < 75 or 105 < angle < 160:
                lines.append((x1, y1, x2, y2))
    return edges, np.array(lines)

def estimate_vanishing_point(lines, img_shape):
    H, W = img_shape[:2]
    def line_intersection(l1, l2):
        x1, y1, x2, y2 = l1; x3, y3, x4, y4 = l2
        denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
        if abs(denom) < 1e-6: return None
        t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
        return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))
    if len(lines) < 2: return (0.5, 0.5), len(lines)
    pts = [p for i in range(len(lines)) for j in range(i + 1, len(lines))
           if (p := line_intersection(lines[i], lines[j])) and -W < p[0] < 2 * W and -H < p[1] < 2 * H]
    if not pts: return (0.5, 0.5), len(lines)
    xs, ys = np.array([p[0] for p in pts]), np.array([p[1] for p in pts])
    hist, xe, ye = np.histogram2d(xs, ys, bins=[np.linspace(-W, 2*W, 30), np.linspace(-H, 2*H, 30)])
    pi = np.unravel_index(hist.argmax(), hist.shape)
    return (((xe[pi[0]] + xe[pi[0]+1]) / 2) / W, ((ye[pi[1]] + ye[pi[1]+1]) / 2) / H), len(lines)

def get_robust_seed_color(hsv, H, W):
    pts = [(W//2, int(H*.95)), (W//2, int(H*.85)), (int(W*.35), int(H*.92)),
           (int(W*.65), int(H*.92)), (int(W*.35), int(H*.82)), (int(W*.65), int(H*.82))]
    colors, valid = [], []
    for sx, sy in pts:
        patch = hsv[max(0, sy-5):sy+5, max(0, sx-10):sx+10]
        if patch.size > 0:
            colors.append(np.mean(patch, axis=(0, 1))); valid.append((sx, sy))
    colors = np.array(colors)
    med = np.median(colors, axis=0)
    best = np.argmin(np.linalg.norm(colors - med, axis=1))
    return colors[best].astype(np.uint8), valid[best]

def segment_road(img, road_region_frac=0.55):
    H, W = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    seed_color, (sx, sy) = get_robust_seed_color(hsv, H, W)
    tol = np.array([20, 60, 60])
    mask = cv2.inRange(hsv, np.clip(seed_color.astype(int)-tol, 0, 255).astype(np.uint8),
                        np.clip(seed_color.astype(int)+tol, 0, 255).astype(np.uint8))
    mask[:int(H*road_region_frac), :] = 0
    flood = np.zeros((H+2, W+2), dtype=np.uint8)
    cv2.floodFill(mask.copy(), flood, (sx, sy), 255, loDiff=(10,10,10), upDiff=(10,10,10))
    road_mask = (flood[1:-1, 1:-1] * 255).astype(np.uint8)
    px = np.where(road_mask > 0)
    if len(px[0]) < 10:
        return road_mask, {'road_area_frac': 0., 'road_centroid_x': 0.5, 'road_taper': 0.}
    area = len(px[0]) / (H * W)
    cx = np.mean(px[1]) / W
    taper = (np.sum(road_mask[int(H*.9), :] > 0) - np.sum(road_mask[int(H*.7), :] > 0)) / W
    return road_mask, {'road_area_frac': area, 'road_centroid_x': cx, 'road_taper': taper}

yolo_model = YOLO('yolov8s.pt')

fig, axes = plt.subplots(5, len(CLASSES), figsize=(4.2 * len(CLASSES), 18))
for i, cls in enumerate(CLASSES):
    img = examples[cls]
    H, W = img.shape[:2]

    axes[0, i].imshow(img); axes[0, i].set_title(cls, fontsize=13, fontweight='bold'); axes[0, i].axis('off')

    _, hog_img = extract_hog(img)
    axes[1, i].imshow(hog_img, cmap='gray'); axes[1, i].set_title('HOG'); axes[1, i].axis('off')

    results = yolo_model(img, verbose=False)
    axes[2, i].imshow(results[0].plot()); axes[2, i].set_title('YOLO detections'); axes[2, i].axis('off')

    edges, lines = detect_edges_and_lines(img)
    vp, n_lines = estimate_vanishing_point(lines, img.shape)
    img_vp = img.copy()
    for x1, y1, x2, y2 in lines: cv2.line(img_vp, (x1, y1), (x2, y2), (0, 255, 0), 1)
    vpx = (int(vp[0]*W), int(vp[1]*H))
    if 0 <= vpx[0] < W and 0 <= vpx[1] < H: cv2.circle(img_vp, vpx, 6, (255, 0, 0), -1)
    axes[3, i].imshow(img_vp); axes[3, i].set_title(f'Hough+VP ({n_lines} lines)'); axes[3, i].axis('off')

    road_mask, feats = segment_road(img)
    overlay = img.copy()
    overlay[road_mask > 0] = (overlay[road_mask > 0]*0.5 + np.array([0,255,0])*0.5).astype(np.uint8)
    axes[4, i].imshow(overlay)
    axes[4, i].set_title(f'area={feats["road_area_frac"]:.2f} cx={feats["road_centroid_x"]:.2f}')
    axes[4, i].axis('off')

plt.suptitle('Sample Frame + Feature Visualization by Maneuver Class', fontsize=16)
plt.tight_layout()
plt.savefig('outputs/class_feature_panel_full.png', dpi=150)
plt.show()

## Update Hough + VP


In [ ]:
CENTER_SEG = (128, 256)  # front-center camera segment, confirmed via seam detection above

def _line_intersection(l1, l2):
    x1, y1, x2, y2 = l1
    x3, y3, x4, y4 = l2
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(denom) < 1e-6:
        return None
    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
    return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))

def _detect_lines_in_roi(gray_center, roi_top_row, canny_low=30, canny_high=100,
                          hough_threshold=12, min_line_length=15, max_line_gap=15):
    H, W = gray_center.shape
    mask = np.zeros_like(gray_center); mask[roi_top_row:, :] = 1
    edges = cv2.Canny(gray_center, canny_low, canny_high)
    edges_masked = edges * mask
    lines_raw = cv2.HoughLinesP(edges_masked, rho=1, theta=np.pi / 180, threshold=hough_threshold,
                                 minLineLength=min_line_length, maxLineGap=max_line_gap)
    lines = []
    if lines_raw is not None:
        for line in lines_raw:
            x1, y1, x2, y2 = line[0]
            angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
            if 20 < angle < 75 or 105 < angle < 160:
                lines.append((x1, y1, x2, y2))
    return np.array(lines)

def _estimate_vp(lines, W, H):
    fallback = (W / 2, H / 2)  # pixel-space center; caller normalizes by W/H
    if len(lines) < 2:
        return fallback, []
    intersections, pair_idx = [], []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            pt = _line_intersection(lines[i], lines[j])
            if pt is not None:
                x, y = pt
                if -W < x < 2 * W and -H < y < 2 * H:
                    intersections.append((x, y)); pair_idx.append((i, j))
    if not intersections:
        return fallback, []
    xs = np.array([p[0] for p in intersections]); ys = np.array([p[1] for p in intersections])
    hist, xe, ye = np.histogram2d(xs, ys, bins=[np.linspace(-W, 2*W, 30), np.linspace(-H, 2*H, 30)])
    pi = np.unravel_index(hist.argmax(), hist.shape)
    vp_x = (xe[pi[0]] + xe[pi[0]+1]) / 2
    vp_y = (ye[pi[1]] + ye[pi[1]+1]) / 2
    return (vp_x, vp_y), list(zip(intersections, pair_idx))

def extract_road_v2(img, inlier_tol_frac=0.15):
    x0, x1 = CENTER_SEG
    center_img = img[:, x0:x1]
    gray = cv2.cvtColor(center_img, cv2.COLOR_RGB2GRAY)
    H, W = gray.shape

    # pass 1: coarse VP estimate, generous fixed ROI (bottom 55%), same starting point as v1
    lines_1 = _detect_lines_in_roi(gray, roi_top_row=int(H * 0.55))
    (vp1_x, vp1_y), _ = _estimate_vp(lines_1, W, H)

    # pass 2: refine ROI using the coarse VP -- but only ever let this GROW the search region
    # relative to the fixed 0.55 fraction, never shrink it (unconditional shrinking compounds a
    # noisy pass-1 estimate into an even worse pass-2 ROI on these short/wide frames)
    margin = int(0.15 * H)
    fixed_top_row = int(H * 0.55)
    vp_informed_top_row = int(np.clip(vp1_y - margin, 0, H - 5))
    roi_top_row_2 = min(fixed_top_row, vp_informed_top_row)
    lines_2 = _detect_lines_in_roi(gray, roi_top_row=roi_top_row_2)
    (vp2_x, vp2_y), intersections_with_idx = _estimate_vp(lines_2, W, H)

    # inlier filtering: keep lines whose intersection with at least one other line lands within
    # inlier_tol_frac of the image diagonal from the final VP estimate
    tol = inlier_tol_frac * np.hypot(W, H)
    inlier_line_idx = set()
    for (ix, iy), (i, j) in intersections_with_idx:
        if np.hypot(ix - vp2_x, iy - vp2_y) < tol:
            inlier_line_idx.add(i); inlier_line_idx.add(j)
    n_lines = len(lines_2)
    n_inliers = len(inlier_line_idx)
    inlier_ratio = n_inliers / n_lines if n_lines > 0 else 0.0

    vp_norm = (vp2_x / W, vp2_y / H)
    return vp_norm, n_lines, n_inliers, inlier_ratio, lines_2, inlier_line_idx, roi_top_row_2

def build_road_v2_features(img):
    vp, n_lines, n_inliers, inlier_ratio, *_ = extract_road_v2(img)
    road_mask, road_feats = segment_road(img)  # segment_road() defined in Section 1's cell above
    H, W = img.shape[:2]
    wb = np.sum(road_mask[int(H*.9), :] > 0) / W
    wm = np.sum(road_mask[int(H*.7), :] > 0) / W
    return np.array([
        vp[0], vp[1], n_lines, n_inliers, inlier_ratio,
        road_feats['road_area_frac'], road_feats['road_centroid_x'], wb, wm,
        road_feats['road_taper'],
    ], dtype=np.float32)

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.2 * len(CLASSES), 4.5))
for i, cls in enumerate(CLASSES):
    img = examples[cls]
    H, W = img.shape[:2]
    x0, x1 = CENTER_SEG
    vp, n_lines, n_inliers, inlier_ratio, lines, inlier_idx, roi_row = extract_road_v2(img)

    img_vis = img.copy()
    for k, (lx1, ly1, lx2, ly2) in enumerate(lines):
        color = (0, 255, 0) if k in inlier_idx else (255, 165, 0)  # green=inlier, orange=rejected
        cv2.line(img_vis, (int(lx1) + x0, int(ly1)), (int(lx2) + x0, int(ly2)), color, 1)
    cv2.line(img_vis, (x0, roi_row), (x1, roi_row), (0, 200, 255), 1)  # adaptive ROI boundary
    vp_px = (int(vp[0] * (x1 - x0)) + x0, int(vp[1] * H))
    if 0 <= vp_px[0] < W and 0 <= vp_px[1] < H:
        cv2.circle(img_vis, vp_px, 6, (255, 0, 0), -1)

    axes[i].imshow(img_vis)
    axes[i].set_title(f'{cls}\nvp=({vp[0]:.2f},{vp[1]:.2f})\n{n_inliers}/{n_lines} inlier lines', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Road Geometry v2: adaptive ROI (cyan line) + inlier filtering (green=kept, orange=rejected)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/road_v2_visualization.png', dpi=140)
plt.show()

In [ ]:
t0 = time.time()
all_road_v2 = []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    all_road_v2.append(build_road_v2_features(img))

road_v2_feats = np.array(all_road_v2)
np.save(os.path.join(FEATURES_DIR, 'road_v2.npy'), road_v2_feats)
print(f'road_v2 shape: {road_v2_feats.shape}  ({time.time()-t0:.0f}s total)')
print('Re-run the Section 2 and Section 5 cells above -- they will pick up road_v2.npy automatically.')

## Vehicles in frame

In [ ]:
VEHICLE_CLASSES = {2: 'car', 5: 'bus', 7: 'truck'}  # subset of DRIVING_CLASSES relevant to lane occupancy
OCC_CENTER_SEG = (128, 256)  # front-center camera segment, same as Section 8

def extract_vehicle_occupancy(img):
    H, W = img.shape[:2]
    x0, x1 = OCC_CENTER_SEG
    seg_w = x1 - x0
    results = yolo_model(img, verbose=False)

    left = []   # list of (weight,) for vehicles in the left third of the segment
    right = []  # list of (weight,) for vehicles in the right third of the segment

    for box in results[0].boxes:
        cls_id = int(box.cls)
        if cls_id not in VEHICLE_CLASSES:
            continue
        bx1, by1, bx2, by2 = box.xyxy[0].tolist()
        cx = (bx1 + bx2) / 2
        if not (x0 <= cx < x1):
            continue  # outside the front-center segment -- different camera, skip
        rel_x = (cx - x0) / seg_w      # 0..1 within the segment
        dx = rel_x - 0.5               # -0.5 (left edge) .. +0.5 (right edge)
        bottom_y = by2 / H             # distance proxy: larger = closer to the vehicle
        weight = bottom_y

        if dx < -1/6:
            left.append(weight)
        elif dx > 1/6:
            right.append(weight)
        # detections within +/-1/6 of center are the ego's own lane, not an adjacent one -- excluded,
        # since the question this feature answers is "can I move into the lane beside me"

    left_count, right_count = len(left), len(right)
    left_crowding = sum(left) if left else 0.0
    right_crowding = sum(right) if right else 0.0
    left_nearest = max(left) if left else 0.0
    right_nearest = max(right) if right else 0.0
    asymmetry = left_crowding - right_crowding

    return np.array([left_count, left_crowding, left_nearest,
                      right_count, right_crowding, right_nearest, asymmetry], dtype=np.float32)

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(4.2 * len(CLASSES), 4.5))
for i, cls in enumerate(CLASSES):
    img = examples[cls]
    H, W = img.shape[:2]
    x0, x1 = OCC_CENTER_SEG
    feat = extract_vehicle_occupancy(img)
    left_count, left_crowding, left_nearest, right_count, right_crowding, right_nearest, asymmetry = feat

    results = yolo_model(img, verbose=False)
    img_vis = img.copy()
    for box in results[0].boxes:
        cls_id = int(box.cls)
        if cls_id not in VEHICLE_CLASSES:
            continue
        bx1, by1, bx2, by2 = [int(v) for v in box.xyxy[0].tolist()]
        cx = (bx1 + bx2) / 2
        if not (x0 <= cx < x1):
            color = (128, 128, 128)
        else:
            dx = (cx - x0) / (x1 - x0) - 0.5
            color = (0, 255, 0) if dx < -1/6 else ((255, 0, 0) if dx > 1/6 else (255, 255, 0))
        cv2.rectangle(img_vis, (bx1, by1), (bx2, by2), color, 2)
    cv2.line(img_vis, (x0, 0), (x0, H), (0, 200, 255), 1)
    cv2.line(img_vis, (x1, 0), (x1, H), (0, 200, 255), 1)

    axes[i].imshow(img_vis)
    axes[i].set_title(f'{cls}\nL: n={int(left_count)} crowd={left_crowding:.2f}\n'
                       f'R: n={int(right_count)} crowd={right_crowding:.2f}  asym={asymmetry:+.2f}',
                       fontsize=8)
    axes[i].axis('off')

plt.suptitle('Vehicle occupancy: green=left-lane vehicle, red=right-lane vehicle, yellow=ego lane (excluded), gray=outside front-center segment', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/vehicle_occupancy_visualization.png', dpi=140)
plt.show()

In [ ]:
t0 = time.time()
all_occ = []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    all_occ.append(extract_vehicle_occupancy(img))

vehicle_occupancy = np.array(all_occ)
np.save(os.path.join(FEATURES_DIR, 'vehicle_occupancy.npy'), vehicle_occupancy)
print(f'vehicle_occupancy shape: {vehicle_occupancy.shape}  ({time.time()-t0:.0f}s total)')
print('Re-run the Section 2 / Section 5 cells above -- they will pick up vehicle_occupancy.npy automatically.')

## 2. t-SNE and UMAP alongside PCA

The existing `pca_visualization.png` cell only looks at linear structure. t-SNE and UMAP can reveal
nonlinear clusters PCA would miss — useful for sanity-checking whether a feature is *really* uninformative
for maneuver class, or just not linearly so.

In [ ]:
import os, numpy as np
for fname in ['hog.npy','hsv.npy','yolo.npy','road.npy',
              'cnn_embedding.npy','mobilenetv2_embedding.npy','vit_embedding.npy','labels.npy']:
    p = os.path.join(FEATURES_DIR, fname)
    if os.path.exists(p):
        arr = np.load(p, allow_pickle=True)
        print(f'{fname:<30} shape={arr.shape}')
    else:
        print(f'{fname:<30} MISSING')

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import umap

hog_feats  = np.load(os.path.join(FEATURES_DIR, 'hog.npy'))
hsv_feats  = np.load(os.path.join(FEATURES_DIR, 'hsv.npy'))
yolo_feats = np.load(os.path.join(FEATURES_DIR, 'yolo.npy'))
road_feats = np.load(os.path.join(FEATURES_DIR, 'road.npy'))
road_v2_path = os.path.join(FEATURES_DIR, 'road_v2.npy')
road_v2_feats = np.load(road_v2_path) if os.path.exists(road_v2_path) else None
labels     = np.load(os.path.join(FEATURES_DIR, 'labels.npy'), allow_pickle=True)

feature_sets = {
    'HOG': hog_feats, 'HSV': hsv_feats, 'YOLO (detections)': yolo_feats,
    'Road Geometry (v1)': road_feats,
    'All Combined': np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats], axis=1),
}
if road_v2_feats is not None:
    feature_sets['Road Geometry (v2)'] = road_v2_feats  # camera-segment-restricted Hough+VP, see Section 8

# include the CNN embedding feature sets once produced (sections 3 and 6)
extra_embeddings = {
    'YOLO-CNN Embedding': 'cnn_embedding.npy',
    'MobileNetV2 Embedding': 'mobilenetv2_embedding.npy',
    'ViT Embedding': 'vit_embedding.npy',
}
loaded_embeddings = {}
for name, fname in extra_embeddings.items():
    path_ = os.path.join(FEATURES_DIR, fname)
    if os.path.exists(path_):
        feats = np.load(path_)
        feature_sets[name] = feats
        loaded_embeddings[name] = feats

if loaded_embeddings:
    feature_sets['All + All Embeddings'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_feats] + list(loaded_embeddings.values()), axis=1)
    print(f'Loaded embeddings: {list(loaded_embeddings.keys())}')

methods = {
    'PCA': lambda X: PCA(n_components=2, random_state=0).fit_transform(X),
    't-SNE': lambda X: TSNE(n_components=2, random_state=0, perplexity=30, init='pca').fit_transform(X),
    'UMAP': lambda X: umap.UMAP(n_components=2, random_state=0).fit_transform(X),
}

fig, axes = plt.subplots(len(methods), len(feature_sets), figsize=(5*len(feature_sets), 5*len(methods)))
for row, (mname, mfunc) in enumerate(methods.items()):
    for col, (fname, feats) in enumerate(feature_sets.items()):
        ax = axes[row, col]
        X = StandardScaler().fit_transform(feats)
        proj = mfunc(X)
        for cls in CLASSES:
            mask = labels == cls
            ax.scatter(proj[mask, 0], proj[mask, 1], c=COLORS[cls], label=cls, alpha=0.4, s=8)
        if row == 0: ax.set_title(fname, fontsize=12)
        if col == 0: ax.set_ylabel(mname, fontsize=12)
        if row == 0 and col == len(feature_sets) - 1:
            ax.legend(fontsize=7, markerscale=2, loc='upper right')
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('PCA vs t-SNE vs UMAP by Feature Set and Maneuver Class', fontsize=15)
plt.tight_layout()
plt.savefig('outputs/embedding_comparison.png', dpi=140)
plt.show()

## 3. New complex feature: dense CNN embedding (YOLOv8s backbone)


In [ ]:
import time

EMBED_LAYER = len(yolo_model.model.model) - 2  # penultimate layer, standard choice per Ultralytics docs

t0 = time.time()
all_cnn = []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    emb = yolo_model.embed(img, verbose=False)[0]
    all_cnn.append(emb.cpu().numpy())

cnn_embedding = np.array(all_cnn)
np.save(os.path.join(FEATURES_DIR, 'cnn_embedding.npy'), cnn_embedding)
print(f'CNN embedding shape: {cnn_embedding.shape}  ({time.time()-t0:.0f}s total)')
print('Re-run the Section 2 cell above -- it will pick up cnn_embedding.npy automatically.')

## SVM/RF baseline

RE RUN ME AFTER NEW FEATURE ADDITION

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cnn_feats = np.load(os.path.join(FEATURES_DIR, 'cnn_embedding.npy'))
road_v2_path = os.path.join(FEATURES_DIR, 'road_v2.npy')
road_v2_feats = np.load(road_v2_path) if os.path.exists(road_v2_path) else None

feature_sets_clf = {
    'HSV': hsv_feats, 'YOLO': yolo_feats, 'Road': road_feats, 'CNN': cnn_feats,
    'HOG': hog_feats,
    'All_Combined': np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats], axis=1),
    'All_CNN': np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats], axis=1),
}
if road_v2_feats is not None:
    # direct v1-vs-v2 test: same combined feature sets, v1's Road swapped for v2's Road_v2
    feature_sets_clf['Road_v2'] = road_v2_feats
    feature_sets_clf['All_Combined_v2Road'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_v2_feats], axis=1)
    feature_sets_clf['All_CNN_v2Road'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_v2_feats, cnn_feats], axis=1)

# pick up MobileNetV2 / ViT embeddings if section 6 has been run
extra_embeddings_clf = {'MNv2': 'mobilenetv2_embedding.npy', 'ViT': 'vit_embedding.npy'}
loaded_extra = {}
for short_name, fname in extra_embeddings_clf.items():
    path_ = os.path.join(FEATURES_DIR, fname)
    if os.path.exists(path_):
        feats = np.load(path_)
        feature_sets_clf[short_name] = feats
        loaded_extra[short_name] = feats

if loaded_extra:
    feature_sets_clf['All_CNN_MNv2_ViT'] = np.concatenate(
        [hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats] + list(loaded_extra.values()), axis=1)
    print(f'Loaded extra embeddings for classification: {list(loaded_extra.keys())}')

majority_baseline = (labels == 'straight').mean()
print(f'Majority-class baseline: {majority_baseline:.3f}\n')

results = {}
best = (None, None, 0.0, None, None)  # feature_set, model, acc, y_test, y_pred

for name, feats in feature_sets_clf.items():
    X_train, X_test, y_train, y_test = train_test_split(
        feats, labels, test_size=0.25, random_state=0, stratify=labels)
    scaler = StandardScaler()
    X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

    svm = SVC(kernel='rbf', class_weight='balanced', random_state=0)
    svm.fit(X_train_s, y_train)
    svm_pred = svm.predict(X_test_s)
    svm_acc = accuracy_score(y_test, svm_pred)

    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=0, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)

    results[name] = dict(svm_acc=svm_acc, rf_acc=rf_acc, rf_importances=rf.feature_importances_,
                          y_test=y_test, svm_pred=svm_pred, rf_pred=rf_pred)
    print(f'{name:14s}  SVM: {svm_acc:.3f}   RF: {rf_acc:.3f}')
    if svm_acc > best[2]: best = (name, 'svm', svm_acc, y_test, svm_pred)
    if rf_acc > best[2]: best = (name, 'rf', rf_acc, y_test, rf_pred)

print(f'\nBest: {best[0]} / {best[1].upper()}  acc={best[2]:.3f}  (majority={majority_baseline:.3f})')

## Confusion matrix + accuracy bar chart + RF importances

In [ ]:
# confusion matrix for the best model + accuracy comparison bar chart
name, model, acc, y_test, y_pred = best
cm = confusion_matrix(y_test, y_pred, labels=CLASSES)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix -- {name} / {model.upper()}\nacc={acc:.3f} (majority baseline={majority_baseline:.3f})')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix_best.png', dpi=150)
plt.show()

print(classification_report(y_test, y_pred, labels=CLASSES, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(feature_sets_clf))
width = 0.35
ax.bar(x - width/2, [results[n]['svm_acc'] for n in feature_sets_clf], width, label='SVM')
ax.bar(x + width/2, [results[n]['rf_acc'] for n in feature_sets_clf], width, label='RF')
ax.axhline(majority_baseline, color='red', linestyle='--', label=f'Majority baseline ({majority_baseline:.3f})')
ax.set_xticks(x); ax.set_xticklabels(list(feature_sets_clf.keys()), rotation=20, ha='right')
ax.set_ylabel('Test accuracy'); ax.set_title('SVM / RF Accuracy by Feature Set vs. Majority Baseline')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/baseline_accuracy_comparison.png', dpi=150)
plt.show()

road_dims = ['vp_x', 'vp_y', 'n_lines', 'road_area_frac', 'road_centroid_x', 'road_width_bottom', 'road_width_mid', 'road_taper']
print('\nRF feature importances -- Road Geometry:')
for dim, imp in sorted(zip(road_dims, results['Road']['rf_importances']), key=lambda t: -t[1]):
    print(f'  {dim:20s} {imp:.3f}')

if 'Road_v2' in results:
    road_v2_dims = ['vp_x', 'vp_y', 'n_lines', 'n_inliers', 'inlier_ratio',
                    'road_area_frac', 'road_centroid_x', 'road_width_bottom', 'road_width_mid', 'road_taper']
    print('\nRF feature importances -- Road Geometry v2:')
    for dim, imp in sorted(zip(road_v2_dims, results['Road_v2']['rf_importances']), key=lambda t: -t[1]):
        print(f'  {dim:20s} {imp:.3f}')

if 'VehicleOcc' in results:
    vehicle_occ_dims = ['left_count', 'left_crowding', 'left_nearest',
                         'right_count', 'right_crowding', 'right_nearest', 'asymmetry']
    print('\nRF feature importances -- Vehicle Occupancy (Section 9):')
    for dim, imp in sorted(zip(vehicle_occ_dims, results['VehicleOcc']['rf_importances']), key=lambda t: -t[1]):
        print(f'  {dim:20s} {imp:.3f}')

## MobileNetV2 + ViT model setup

MobileNetV2 (Sandler et al., 2018) is a CNN designed for efficient inference on phones/embedded devices, and it's the architecture that came out on top in the motion-prediction literature we reviewed (Djuric et al. 2020, Table 1). Its core building block is the inverted residual with linear bottleneck: a 1x1 convolution expands the number of channels, a 3x3 depthwise separable convolution processes each channel independently (spatially), and another 1x1 convolution projects back down to a small number of channels, with a residual (skip) connection linking input to output. This is the reverse of a classic ResNet block (which goes wide → narrow → wide) — hence "inverted." The depthwise separable convolution is the key efficiency trick: a standard convolution mixes space and channels together in one expensive operation, while depthwise separable splits that into a cheap per-channel spatial pass followed by a cheap 1x1 channel-mixing pass, cutting compute by roughly an order of magnitude for similar accuracy. Like all CNNs, it builds up a global understanding of the image gradually — each layer only sees a local neighborhood, and larger patterns emerge by stacking layers. It's pretrained on ImageNet (1.2M images, 1,000 object classes). In our pipeline we strip its final classification layer and global-average-pool the last convolutional feature map, giving a 1,280-dim vector per image.

ViT (Vision Transformer) (Dosovitskiy et al., 2020, "An Image is Worth 16x16 Words") takes a very different approach: instead of convolutions, it applies the Transformer architecture originally built for language directly to images. The image is cut into a grid of fixed-size patches (16x16 pixels for ViT-B/16), each patch is flattened and linearly projected into a vector, a position embedding is added, and a learnable [CLS] token is prepended to the sequence. All of this is fed through standard Transformer encoder layers, where self-attention lets every patch directly attend to every other patch, regardless of distance — so the model can relate the left and right edges of an image in its very first layer, something a CNN only achieves after enough stacked layers to grow its receptive field that wide. The [CLS] token's final representation is treated as a whole-image summary — the 768-dim vector we extract.

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights, vit_b_16, ViT_B_16_Weights
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}, torchvision {torchvision.__version__}, device={device}')

# --- MobileNetV2: strip classifier head, global-average-pool the last conv feature map ---
mnv2 = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).to(device).eval()
mnv2_preprocess = MobileNet_V2_Weights.IMAGENET1K_V1.transforms()

def extract_mobilenetv2(img):
    pil_img = Image.fromarray(img)
    x = mnv2_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = mnv2.features(x)               # (1, 1280, H', W')
        pooled = feats.mean(dim=[2, 3])          # global average pool -> (1, 1280)
    return pooled.squeeze(0).cpu().numpy()

# --- ViT-B/16: use the pre-logits CLS token as the embedding ---
vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).to(device).eval()

def pad_to_square(pil_img):
    w, h = pil_img.size
    size = max(w, h)
    new_img = Image.new('RGB', (size, size), (0, 0, 0))
    new_img.paste(pil_img, ((size - w) // 2, (size - h) // 2))
    return new_img

vit_preprocess = transforms.Compose([
    transforms.Lambda(pad_to_square),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_vit(img):
    pil_img = Image.fromarray(img)
    x = vit_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = vit._process_input(x)
        n = feats.shape[0]
        batch_class_token = vit.class_token.expand(n, -1, -1)
        feats = torch.cat([batch_class_token, feats], dim=1)
        feats = vit.encoder(feats)
        cls_token = feats[:, 0]                  # (1, 768) pre-logits CLS embedding
    return cls_token.squeeze(0).cpu().numpy()

print('MobileNetV2 and ViT-B/16 loaded and ready.')

In [ ]:
import time

t0 = time.time()
all_mnv2, all_vit = [], []
for i, (seq_id, entry) in enumerate(manifest.items()):
    if (i + 1) % 200 == 0:
        print(f'{i+1}/{len(manifest)} ({time.time()-t0:.0f}s)')
    data = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
    img = np.array(data['_modality_data'].item()[Modality.CAMERAS])
    all_mnv2.append(extract_mobilenetv2(img))
    all_vit.append(extract_vit(img))

mnv2_embedding = np.array(all_mnv2)
vit_embedding = np.array(all_vit)
np.save(os.path.join(FEATURES_DIR, 'mobilenetv2_embedding.npy'), mnv2_embedding)
np.save(os.path.join(FEATURES_DIR, 'vit_embedding.npy'), vit_embedding)
print(f'MobileNetV2 embedding: {mnv2_embedding.shape}')
print(f'ViT embedding: {vit_embedding.shape}')
print(f'Total time: {time.time()-t0:.0f}s')
print('Re-run the Section 2 / Section 5 cells above -- they will pick these up automatically once you add them to feature_sets.')

In [ ]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

MINORITY_CLASSES = ['lane-change-left', 'lane-change-right']

# reuse the "All_CNN" feature set (current best: HOG + HSV + YOLO + Road + YOLO-CNN embedding)
feats = np.concatenate([hog_feats, hsv_feats, yolo_feats, road_feats, cnn_feats], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    feats, labels, test_size=0.25, random_state=0, stratify=labels)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
train_counts = Counter(y_train)
print(f'Training class counts: {train_counts}')


def duplicate_with_jitter(X, y, target_classes, multiplier=3, jitter_frac=0.1, random_state=0):
    rng = np.random.RandomState(random_state)
    feature_std = X.std(axis=0)
    feature_std[feature_std == 0] = 1.0
    X_aug, y_aug = [X], [y]
    for cls in target_classes:
        mask = y == cls
        X_cls = X[mask]
        for _ in range(multiplier - 1):
            noise = rng.normal(0, jitter_frac, size=X_cls.shape) * feature_std
            X_aug.append(X_cls + noise)
            y_aug.append(np.array([cls] * len(X_cls)))
    return np.concatenate(X_aug), np.concatenate(y_aug)


def evaluate(name, X_tr, y_tr):
    svm = SVC(kernel='rbf', class_weight='balanced', random_state=0)
    svm.fit(X_tr, y_tr)
    pred = svm.predict(X_test_s)
    acc = accuracy_score(y_test, pred)
    report = classification_report(y_test, pred, labels=CLASSES, zero_division=0, output_dict=True)
    lc_left_recall = report['lane-change-left']['recall']
    lc_right_recall = report['lane-change-right']['recall']
    print(f'\n=== {name} ===')
    print(f'Overall accuracy: {acc:.3f}   lc-left recall: {lc_left_recall:.2f}   lc-right recall: {lc_right_recall:.2f}')
    print(classification_report(y_test, pred, labels=CLASSES, zero_division=0))
    return dict(name=name, acc=acc, lc_left_recall=lc_left_recall, lc_right_recall=lc_right_recall)


results = []
results.append(evaluate('Baseline (class_weight=balanced)', X_train_s, y_train))

sampling_strategy = {cls: train_counts[cls] * 3 for cls in MINORITY_CLASSES}
smote = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=5, random_state=0)
X_smote, y_smote = smote.fit_resample(X_train_s, y_train)
results.append(evaluate('SMOTE (3x lane-change classes)', X_smote, y_smote))

X_dup, y_dup = duplicate_with_jitter(X_train_s, y_train, MINORITY_CLASSES, multiplier=3, jitter_frac=0.1)
results.append(evaluate('Duplication + jitter (3x lane-change classes)', X_dup, y_dup))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = [r['name'].split(' (')[0] for r in results]
axes[0].bar(names, [r['acc'] for r in results], color=['steelblue', 'tomato', 'green'])
axes[0].axhline((labels == 'straight').mean(), color='gray', linestyle='--', label='Majority baseline')
axes[0].set_ylabel('Overall test accuracy'); axes[0].set_title('Overall Accuracy'); axes[0].legend()
axes[0].tick_params(axis='x', rotation=20)

x = np.arange(len(results))
width = 0.35
axes[1].bar(x - width/2, [r['lc_left_recall'] for r in results], width, label='lane-change-left recall')
axes[1].bar(x + width/2, [r['lc_right_recall'] for r in results], width, label='lane-change-right recall')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20)
axes[1].set_ylabel('Recall'); axes[1].set_title('Minority-Class Recall'); axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/imbalance_comparison.png', dpi=150)
plt.show()

## Failures Analysis 

In [ ]:
# --- Capture test-set predictions with sequence IDs for error review ---
import os, numpy as np
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

FEATURES_DIR = '../data/processed/waymo_e2e/features/'
CLASSES = ['straight','left-turn','right-turn','lane-change-left','lane-change-right']

# load the combined feature set + labels + sequence ids
hog  = np.load(os.path.join(FEATURES_DIR,'hog.npy'))
yolo = np.load(os.path.join(FEATURES_DIR,'yolo.npy'))
road = np.load(os.path.join(FEATURES_DIR,'road.npy'))
X_all = np.concatenate([hog, yolo, road], axis=1)
labels  = np.load(os.path.join(FEATURES_DIR,'labels.npy'), allow_pickle=True)
seq_ids = np.load(os.path.join(FEATURES_DIR,'seq_ids.npy'), allow_pickle=True)

# keep only the 5 target classes (drop the rare 'stationary')
keep = np.isin(labels, CLASSES)
X_all, labels, seq_ids = X_all[keep], labels[keep], seq_ids[keep]

# stratified split, tracking indices so we recover sequence ids for the test set
idx = np.arange(len(labels))
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    X_all, labels, idx, test_size=0.25, random_state=0, stratify=labels)

scaler = StandardScaler().fit(Xtr)
clf = SVC(kernel='rbf', random_state=0).fit(scaler.transform(Xtr), ytr)
y_pred = clf.predict(scaler.transform(Xte))

test_seq_ids = seq_ids[ite]
acc = (y_pred == yte).mean()
print(f'Test accuracy: {acc:.3f}   (n_test={len(yte)})')

# collect misclassified per true class
misclassified = {c: [] for c in CLASSES}
for sid, true, pred in zip(test_seq_ids, yte, y_pred):
    if true != pred:
        misclassified[true].append((sid, true, pred))
for c in CLASSES:
    print(f'  {c:<20} {len(misclassified[c])} misclassified')

In [ ]:
# --- Review misclassified examples: filmstrip + trajectory, per class ---
import json, matplotlib.pyplot as plt
from standard_e2e import Modality, TrajectoryComponent

MANIFEST_PATH = '../data/train_manifest.json'
TRAIN_DIR     = '../data/processed/waymo_e2e/training/'
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

def _heading_ends(xs, ys, min_disp=0.5):
    pts = np.column_stack([xs, ys]); sh=eh=None
    for i in range(1,len(pts)):
        if np.hypot(*(pts[i]-pts[0]))>=min_disp:
            sh=np.arctan2(pts[i][1]-pts[0][1],pts[i][0]-pts[0][0]); break
    for i in range(len(pts)-2,-1,-1):
        if np.hypot(*(pts[-1]-pts[i]))>=min_disp:
            eh=np.arctan2(pts[-1][1]-pts[i][1],pts[-1][0]-pts[i][0]); break
    return sh,eh

def review_class(true_class, n_examples=3, n_frames=5):
    items = misclassified.get(true_class, [])[:n_examples]
    if not items:
        print(f'No misclassified examples for {true_class}'); return
    for sid, true, pred in items:
        entry = manifest[sid]
        # all frames for this sequence, in order
        seq_files = sorted(
            [f for f in os.listdir(TRAIN_DIR) if f.startswith(sid)],
            key=lambda f: int(f.rsplit('_',1)[1].replace('.npz','')))
        fidx = np.linspace(0, len(seq_files)-1, n_frames).astype(int)

        # trajectory from the target frame
        d = np.load(os.path.join(TRAIN_DIR, entry['target_fname']), allow_pickle=True)
        fut = d['_modality_data'].item()[Modality.FUTURE_STATES]
        xs = fut.get(TrajectoryComponent.X).flatten()
        ys = fut.get(TrajectoryComponent.Y).flatten()
        dy = ys[-1]-ys[0]; dx = xs[-1]-xs[0]
        sh,eh = _heading_ends(xs,ys)
        hd = 0.0 if (sh is None or eh is None) else np.degrees(np.arctan2(np.sin(eh-sh),np.cos(eh-sh)))

        fig, axes = plt.subplots(1, n_frames+1, figsize=((n_frames+1)*3.0, 2.8))
        for j, fi in enumerate(fidx):
            di = np.load(os.path.join(TRAIN_DIR, seq_files[fi]), allow_pickle=True)
            img = np.array(di['_modality_data'].item()[Modality.CAMERAS])
            axes[j].imshow(img); axes[j].set_title(f'frame {fi}', fontsize=8); axes[j].axis('off')
        ax = axes[-1]
        ax.plot(-ys, xs, '-o', ms=3)
        ax.plot(-ys[0], xs[0], 'go', ms=8); ax.plot(-ys[-1], xs[-1], 'rs', ms=8)
        ax.axvline(0, color='gray', lw=0.5, ls='--'); ax.set_aspect('equal','box')
        ax.set_title(f'traj\nΔhead={hd:.0f}° lat={dy:+.1f}', fontsize=8)
        ax.set_xlabel('← right  left →', fontsize=7); ax.tick_params(labelsize=6)

        fig.suptitle(f'{sid[:10]}   TRUE: {true}   →   PREDICTED: {pred}',
                     fontsize=12, y=1.03)
        plt.tight_layout()
        plt.savefig(f'outputs/miscls_{true_class}_{sid[:8]}.png', dpi=110, bbox_inches='tight')
        plt.show()

# review a few from each class — start with the lane-change failures (the headline)
for c in ['lane-change-left','lane-change-right','left-turn','right-turn','straight']:
    print(f'\n{"="*70}\n{c.upper()} — misclassified\n{"="*70}')
    review_class(c, n_examples=3)